# Loan Repayment — PD Model Benchmark (Colab)

End-to-end run of the `tht` package on Colab: install from GitHub, load data from Drive, then benchmark logistic / HistGradientBoosting / LightGBM / XGBoost.

In [ ]:
# 0. Extra deps (idempotent).
!pip install -q lightgbm xgboost

In [ ]:
# 1. Install `tht` straight from GitHub (this is what makes `import tht` work).
!pip install git+https://github.com/asadullahqb/loan-repayment-challenge.git

In [ ]:
# 2. Environment bootstrap: mount Drive for data + sys.path fallback (no-op locally).
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = "https://github.com/asadullahqb/loan-repayment-challenge.git"
    REPO_DIR = Path("/content/loan-repayment-challenge")

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", REPO, str(REPO_DIR)], check=True)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)

    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["THT_DATA_DIR"] = "/content/drive/MyDrive/tht/data/raw/parquet"

# Fallback: add the repo's src/ to sys.path if the editable install isn't visible.
try:
    import tht  # noqa: F401
except ModuleNotFoundError as e:
    if e.name != "tht":
        raise
    for _c in (
        Path.cwd() / "src",
        Path.cwd().parent / "src",
        Path("/content/loan-repayment-challenge/src"),
    ):
        if (_c / "tht" / "__init__.py").exists():
            sys.path.insert(0, str(_c))
            break
    import tht  # noqa: F401

print("tht imported from:", tht.__file__)

## 3. Load data (from Drive, via `THT_DATA_DIR` set above)

In [ ]:
data = tht.load_data()
print({k: v.shape for k, v in data.items()})

## 4. Build features + benchmark (full data)

`run_benchmark(..., tune=False)` uses fixed hyperparameters (fast). Set `tune=True` for a small grid search (4 combos x 3-fold CV per GBM).

In [ ]:
X, y = tht.build_features(data["loan"], data["payment"], data["underwriting"], behavioral=False)

print(f"Resolved cohort : {len(y):,} loans")
print(f"Default rate    : {y.mean():.1%}")
print(f"Features        : {X.shape[1]}\n")

metrics, importances = tht.run_benchmark(X, y, tune=False)  # set tune=True for a bit more

print("Test-set metrics (30% holdout, stratified):")
print(f"{'model':<14}{'AUC':>8}{'KS':>8}{'Gini':>8}")
for name, m in metrics.items():
    print(f"{name:<14}{m['auc']:>8.3f}{m['ks']:>8.3f}{m['gini']:>8.3f}")

print("\nLightGBM permutation importances (top features):")
for feat, imp in importances.items():
    print(f"  {feat:<24}{imp:>8.4f}")

## 5. Optional: quick smoke test on a sample

In [ ]:
Xs, ys = tht.build_features(data["loan"], data["payment"], data["underwriting"], sample_frac=0.2)
metrics_s, _ = tht.run_benchmark(Xs, ys, tune=False)
print({k: round(v['auc'], 3) for k, v in metrics_s.items()})